# Wrong-way detection on a Colab GPU

Runs `pipeline.py` (RF-DETR + ByteTrack + `WrongWayDetector`) against
traffic video.

Works either in the browser at colab.research.google.com, or in VS Code
through the official Google Colab extension. Either way the code executes
on a **remote Google machine**, which cannot see your local disk -- so
the cells below fetch the code from GitHub and the video from either the
`supervision` sample set or Drive, rather than assuming anything is
already there.

Pick a GPU runtime before running. The free tier gives a T4; a Colab Pro
subscription unlocks faster ones (L4, A100), and the extension can use
those too.

**Read the class-id table in the run cell's output before you read any
alert.** It settles the open question in `CLAUDE.md`: whether
`VEHICLE_CLASS_IDS = {2, 3, 5, 7}` matches this RF-DETR build. If the
names beside those ids are not vehicles, every alert below them is
meaningless.

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("No GPU. Switch the runtime to a GPU type, then rerun this cell.")

CUDA available: False
No GPU. Switch the runtime to a GPU type, then rerun this cell.


## 1. Install

Only two packages. The runtime already ships `torch` (CUDA build),
`opencv` and `numpy`; installing `requirements.txt` wholesale can replace
the CUDA torch with a CPU one and silently cost you the GPU.
`supervision` arrives as a dependency of `rfdetr`.

In [2]:
!pip install -q rfdetr trackers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.1/491.1 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.8/220.8 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.3/373.3 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 52.4 MB/s eta 0:00:00:00:0100:01


## 2. Get the code

Clones on the first run of a session, pulls on every run after that.
**Re-run this cell after every push** — it is what carries your local
edits across to the machine that actually executes them.

It also enables `autoreload`. Without it, pulling new code changes the
files on disk but not what the kernel runs: Python caches imported modules
in memory, so a fixed bug keeps reproducing and the fix looks like it
failed. That is worth knowing even outside this notebook — it is the most
common reason "I already fixed that" turns out to be false in any Jupyter
session.

In [ ]:
# Python caches imported modules, so `git pull` alone does not change what
# a running kernel executes -- it keeps running the version already in
# memory. autoreload re-reads changed files before each cell, which is what
# makes edit -> push -> pull -> run actually work without a kernel restart.
%load_ext autoreload
%autoreload 2

import os

REPO_URL = "https://github.com/Arielevi15/Crime_Traffic_Dedector.git"
REPO_DIR = "/content/Crime_Traffic_Dedector"

if not os.path.isdir(REPO_DIR):
    !git clone --quiet {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git pull --ff-only
!ls

## 3. Get the video

Two options. Run **3a** for the very first run, and **3b** once you have
real footage -- they answer different questions.

### 3a. Sample footage (start here)

One line, no upload, no Drive. `supervision` ships it, and it is already
installed as a dependency of `rfdetr`.

Be clear about what this does and does not prove. It is **elevated
highway footage, not a forward-facing dashcam**, so it does not match the
scope assumption at the top of `CLAUDE.md`. Good for: confirming the
class ids, that RF-DETR loads, that ByteTrack holds ids across frames,
and that the chain runs end to end. Not good for: tuning any threshold in
`DetectorConfig`, or judging the wrong-way logic in the domain we
actually care about.

In [4]:
from supervision.assets import VideoAssets, download_assets

VIDEO = download_assets(VideoAssets.VEHICLES)
print("Video ready:", VIDEO)

[2026-08-06 18:50:43] [INFO] supervision.assets.downloader - Downloading vehicles.mp4 assets


  0%|          | 0/35345757 [00:00<?, ?it/s]

Video ready: vehicles.mp4


### 3b. Your own dashcam footage

Skip this on the first run. Once you have real clips, put them in a Drive
folder once and every future session sees them without another upload.
Mounting opens an auth prompt the first time. Running this cell
overwrites `VIDEO` from 3a.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Point this at your own clip. Not sure of the exact path? Find it:
#   !find /content/drive/MyDrive -iname "*.mp4" | head -20
# Note that Drive's "Shared with me" is a view, not a folder -- it is
# never mounted. Add a shortcut to My Drive, or copy the file there.
candidate = "/content/drive/MyDrive/Traffic_Crimes_Model_Detector_Project/Traffic_Crimes_Model_Detector_Project/Dashcam/sample1.mp4"

# Check first, assign second. Assigning before the check would let a
# wrong path here silently replace a working VIDEO set by cell 3a, and
# the failure would then surface much later, in the run cell.
assert os.path.isfile(candidate), (
    "Not found: {0}\nRun the find command above to check the name.".format(candidate)
)
VIDEO = candidate
print("Video ready:", VIDEO)

Mounted at /content/drive


AssertionError: Not found: /content/drive/MyDrive/dashcam/sample1.mp4
Run the find command above to check the name.

## 4. Run

`limit_frames=300` keeps the first run short: long enough to produce the
class-id table and show whether tracking holds, short enough that a
misconfiguration costs seconds rather than an hour. Drop it once the
class ids are confirmed.

In [ ]:
from pipeline import run

alerts = run(
    video=VIDEO,
    output="check.mp4",
    limit_frames=300,
    # Records what the violation modules actually consume. See section 7 --
    # this is what makes logic debugging fast.
    dump_tracks="tracks.jsonl",
)
alerts

## 5. Watch the annotated result

Green box = tracked vehicle, red = alerted, orange dot = the
road-contact point the detector actually reasons about. If those dots are
not landing on the road beneath each vehicle, fix that before tuning any
threshold -- everything downstream depends on that point being right.

OpenCV writes `mp4v`, which the notebook player will not decode, so
re-encode to H.264 first. The video is inlined as base64, which is fine
for a few hundred frames; for a full clip, download it instead.

In [ ]:
from base64 import b64encode

from IPython.display import HTML

!ffmpeg -loglevel error -i check.mp4 -vcodec libx264 -y check_h264.mp4

payload = b64encode(open("check_h264.mp4", "rb").read()).decode()
HTML('<video width=720 controls><source src="data:video/mp4;base64,{0}">'.format(payload))

In [ ]:
# Longer clips: copy the result back to Drive instead of inlining it.
!cp check_h264.mp4 /content/drive/MyDrive/dashcam/

## 6. Tuning

Once the class ids are confirmed, the next open task is tuning
`DetectorConfig` against real footage. Pass one in explicitly rather than
editing the module, so the tested defaults stay intact:

```python
from wrong_way_detector import DetectorConfig

alerts = run(
    video=VIDEO,
    output="check.mp4",
    config=DetectorConfig(opposite_cos_threshold=-0.6),
)
```

Per `CLAUDE.md` principle 3, tune toward silence. A false positive
accuses an innocent driver; a false negative merely misses one.

## 7. Take the track data home — stop iterating through the GPU

The run cell wrote `tracks.jsonl`: per frame, every track's id and
road-contact point. That is the entire input surface a violation module
has (`CLAUDE.md` principle 5), so **everything downstream of perception
can be reproduced from it exactly** — with no GPU, no model, no video and
no third-party packages.

This matters because of arithmetic. Iterating on the logic through this
notebook costs minutes per attempt: edit, push, pull, reload the model,
decode video. Replaying the same run locally costs under a second. When
the thing being debugged is the logic rather than the perception — which
is nearly always — there is no reason to pay the GPU cost again.

Download the file, drop it in the project folder, and work locally:

```
python replay.py tracks.jsonl
python replay.py tracks.jsonl --track 16 --verbose
python replay.py tracks.jsonl --zone-size 240 --opposite-cos-threshold -0.6
```

`--track N --verbose` prints that one vehicle's per-frame decision trail —
heading, zone, whether the zone was trusted, cosine, streak. It is the
fastest way to see why a specific alert fired. The threshold flags let a
parameter sweep be a shell loop instead of a code edit.

In [ ]:
import os

print("tracks.jsonl:", os.path.getsize("tracks.jsonl") // 1024, "KB")

from google.colab import files

files.download("tracks.jsonl")